In [1]:
SEED = 42
import numpy as np
import torch
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
from pathlib import Path
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
import sys
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

DATA_DIR   = Path("../data/processed/FD001")
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X      = np.load(DATA_DIR / "X_train.npy")   # (N, 30, 14)
y      = np.load(DATA_DIR / "y_train.npy")   # (N,)
labels = np.load(DATA_DIR / "labels_train.npy")  # (N,) int 0-3

print(f"X      : {X.shape}")
print(f"y      : {y.shape}")
print(f"labels : {labels.shape}  classes: {np.unique(labels)}")
print(f"Device : {DEVICE}")


X      : (17731, 30, 17)
y      : (17731,)
labels : (17731,)  classes: [0 1 2 3]
Device : cpu


In [2]:
SEQ_LEN    = 30       
INPUT_DIM  = 14       
HIDDEN_DIM = 24       
LATENT_DIM = 24       
NUM_LAYERS = 3       
NUM_CLASSES= 4        
EMBED_DIM  = 8        

print("Hyperparameters set.")


Hyperparameters set.


In [3]:
class Embedder(nn.Module):
    """
    Maps real sequences X → latent H.
    Architecture: GRU + linear projection
    Input : (batch, seq_len, input_dim)
    Output: (batch, seq_len, hidden_dim)
    """
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h, _ = self.gru(x)          # (B, T, hidden_dim)
        return self.proj(h)          # (B, T, hidden_dim)


embedder = Embedder(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
print(embedder)
print(f"\nParameters: {sum(p.numel() for p in embedder.parameters()):,}")


Embedder(
  (gru): GRU(14, 24, num_layers=3, batch_first=True)
  (proj): Sequential(
    (0): Linear(in_features=24, out_features=24, bias=True)
    (1): Sigmoid()
  )
)

Parameters: 10,680


In [4]:
class Recovery(nn.Module):
    """
    Maps latent H -> reconstructed X.
    Input : (batch, seq_len, hidden_dim)
    Output: (batch, seq_len, input_dim)
    """
    def __init__(self, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

    def forward(self, h):
        out, _ = self.gru(h)
        return self.proj(out)        # (B, T, input_dim)


recovery = Recovery(HIDDEN_DIM, INPUT_DIM, NUM_LAYERS).to(DEVICE)
print(recovery)
print(f"\nParameters: {sum(p.numel() for p in recovery.parameters()):,}")


Recovery(
  (gru): GRU(24, 24, num_layers=3, batch_first=True)
  (proj): Sequential(
    (0): Linear(in_features=24, out_features=14, bias=True)
    (1): Sigmoid()
  )
)

Parameters: 11,150


In [5]:
class Supervisor(nn.Module):
    """
    Enforces temporal dependency in latent space.
    Trained to predict H_{t+1} given H_{1:t}.
    Input : (batch, seq_len, hidden_dim)
    Output: (batch, seq_len, hidden_dim)
    """
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers - 1,   # shallower than embedder
            batch_first=True,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
        )

    def forward(self, h):
        out, _ = self.gru(h)
        return self.proj(out)


supervisor = Supervisor(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
print(supervisor)
print(f"\nParameters: {sum(p.numel() for p in supervisor.parameters()):,}")


Supervisor(
  (gru): GRU(24, 24, num_layers=2, batch_first=True)
  (proj): Sequential(
    (0): Linear(in_features=24, out_features=24, bias=True)
    (1): Sigmoid()
  )
)

Parameters: 7,800


In [6]:
class Generator(nn.Module):
    """
    Conditional generator: noise Z + class label -> latent H_hat.
    Condition vector (one-hot) is concatenated to noise at every timestep.
    Input : Z (batch, seq_len, latent_dim), c (batch, num_classes)
    Output: (batch, seq_len, hidden_dim)
    """
    def __init__(self, latent_dim, hidden_dim, num_layers, num_classes, embed_dim):
        super().__init__()
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.gru = nn.GRU(
            input_size=latent_dim + embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
        )

    def forward(self, z, c):
        # c: (B,) integer class indices
        emb = self.class_embed(c)                      # (B, embed_dim)
        emb = emb.unsqueeze(1).repeat(1, z.size(1), 1) # (B, T, embed_dim)
        inp = torch.cat([z, emb], dim=-1)               # (B, T, latent+embed)
        h, _ = self.gru(inp)
        return self.proj(h)                             # (B, T, hidden_dim)


generator = Generator(LATENT_DIM, HIDDEN_DIM, NUM_LAYERS, NUM_CLASSES, EMBED_DIM).to(DEVICE)
print(generator)
print(f"\nParameters: {sum(p.numel() for p in generator.parameters()):,}")


Generator(
  (class_embed): Embedding(4, 8)
  (gru): GRU(32, 24, num_layers=3, batch_first=True)
  (proj): Sequential(
    (0): Linear(in_features=24, out_features=24, bias=True)
    (1): Sigmoid()
  )
)

Parameters: 12,008


In [7]:
class Discriminator(nn.Module):
    """
    Classifies latent sequences as real or synthetic.
    Conditional: class embedding is injected at input.
    Input : H (batch, seq_len, hidden_dim), c (batch,)
    Output: (batch, seq_len, 1)  -- sequence-level logits
    """
    def __init__(self, hidden_dim, num_layers, num_classes, embed_dim):
        super().__init__()
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.gru = nn.GRU(
            input_size=hidden_dim + embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.proj = nn.Linear(hidden_dim, 1)   # no sigmoid — use BCEWithLogitsLoss

    def forward(self, h, c):
        emb = self.class_embed(c)
        emb = emb.unsqueeze(1).repeat(1, h.size(1), 1)
        inp = torch.cat([h, emb], dim=-1)
        out, _ = self.gru(inp)
        return self.proj(out)                   # (B, T, 1)


discriminator = Discriminator(HIDDEN_DIM, NUM_LAYERS, NUM_CLASSES, EMBED_DIM).to(DEVICE)
print(discriminator)
print(f"\nParameters: {sum(p.numel() for p in discriminator.parameters()):,}")


Discriminator(
  (class_embed): Embedding(4, 8)
  (gru): GRU(32, 24, num_layers=3, batch_first=True)
  (proj): Linear(in_features=24, out_features=1, bias=True)
)

Parameters: 11,433


In [8]:
BATCH = 8

x_real = torch.randn(BATCH, SEQ_LEN, INPUT_DIM).to(DEVICE)
z      = torch.randn(BATCH, SEQ_LEN, LATENT_DIM).to(DEVICE)
c      = torch.randint(0, NUM_CLASSES, (BATCH,)).to(DEVICE)

with torch.no_grad():
    H        = embedder(x_real)          # (B, T, hidden)
    X_rec    = recovery(H)               # (B, T, input_dim)
    H_sup    = supervisor(H)             # (B, T, hidden)
    H_hat    = generator(z, c)           # (B, T, hidden)
    H_sup_g  = supervisor(H_hat)         # (B, T, hidden)
    X_hat    = recovery(H_hat)           # (B, T, input_dim)  <- synthetic sequence
    D_real   = discriminator(H, c)       # (B, T, 1)
    D_fake   = discriminator(H_hat, c)   # (B, T, 1)

print("Embedder  out :", H.shape)
print("Recovery  out :", X_rec.shape)
print("Supervisor out:", H_sup.shape)
print("Generator out :", H_hat.shape)
print("Synthetic X   :", X_hat.shape)
print("D real        :", D_real.shape)
print("D fake        :", D_fake.shape)
print("\nAll forward passes OK.")


Embedder  out : torch.Size([8, 30, 24])
Recovery  out : torch.Size([8, 30, 14])
Supervisor out: torch.Size([8, 30, 24])
Generator out : torch.Size([8, 30, 24])
Synthetic X   : torch.Size([8, 30, 14])
D real        : torch.Size([8, 30, 1])
D fake        : torch.Size([8, 30, 1])

All forward passes OK.


In [9]:

bce_logits = nn.BCEWithLogitsLoss()
mse        = nn.MSELoss()


def loss_autoencoder(x, x_recon):
    """Phase 1: embedder + recovery reconstruction loss."""
    return 10.0 * torch.sqrt(mse(x, x_recon))


def loss_supervisor(h, h_sup):
    """Phase 1: supervisor next-step prediction loss."""
    return 10.0 * torch.sqrt(mse(h[:, 1:, :], h_sup[:, :-1, :]))


def loss_generator(d_fake, h, h_hat_sup, x, x_hat, gamma=1.0):
    """
    Phase 3 joint GAN loss for generator.
    L_G = L_GAN + gamma * L_moments + L_supervised
    """
    ones  = torch.ones_like(d_fake)
    L_GAN = bce_logits(d_fake, ones)

    # Supervised loss: generator should match real latent dynamics
    L_sup = 10.0 * torch.sqrt(mse(h[:, 1:, :], h_hat_sup[:, :-1, :]))

    # Moments loss: match mean and variance per feature
    L_mean = torch.mean(
        torch.abs(torch.mean(x, dim=0) - torch.mean(x_hat, dim=0))
    )
    L_var  = torch.mean(
        torch.abs(torch.sqrt(torch.var(x, dim=0) + 1e-6)
                  - torch.sqrt(torch.var(x_hat, dim=0) + 1e-6))
    )
    L_moments = L_mean + L_var

    return L_GAN + gamma * L_moments + L_sup


def loss_discriminator(d_real, d_fake, gamma=1.0):
    """Phase 3 discriminator loss."""
    ones  = torch.ones_like(d_real)
    zeros = torch.zeros_like(d_fake)
    L_real = bce_logits(d_real, ones)
    L_fake = bce_logits(d_fake, zeros)
    return (L_real + L_fake) / 2.0


print("Loss functions defined.")


Loss functions defined.


In [10]:

models = {
    "Embedder"      : embedder,
    "Recovery"      : recovery,
    "Supervisor"    : supervisor,
    "Generator"     : generator,
    "Discriminator" : discriminator,
}

total = 0
for name, model in models.items():
    n = sum(p.numel() for p in model.parameters())
    total += n
    print(f"{name:<16} {n:>10,} params")

print("-" * 30)
print(f"{'Total':<16} {total:>10,} params")


Embedder             10,680 params
Recovery             11,150 params
Supervisor            7,800 params
Generator            12,008 params
Discriminator        11,433 params
------------------------------
Total                53,071 params


In [11]:
import json

arch_config = {
    "seq_len"    : SEQ_LEN,
    "input_dim"  : INPUT_DIM,
    "hidden_dim" : HIDDEN_DIM,
    "latent_dim" : LATENT_DIM,
    "num_layers" : NUM_LAYERS,
    "num_classes": NUM_CLASSES,
    "embed_dim"  : EMBED_DIM,
}

config_path = Path("../configs/model_config.json")
config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as f:
    json.dump(arch_config, f, indent=2)

print(f"Architecture config saved -> {config_path.resolve()}")
print(json.dumps(arch_config, indent=2))


Architecture config saved -> C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\configs\model_config.json
{
  "seq_len": 30,
  "input_dim": 14,
  "hidden_dim": 24,
  "latent_dim": 24,
  "num_layers": 3,
  "num_classes": 4,
  "embed_dim": 8
}
